# Persona 2 — Del PDF al texto estructurado (Glaciares · v2)
**Proyecto:** Enterprise Brain — Gestión del Conocimiento con Knowledge Graph + GraphRAG · Grupo G8

**Entrada** (Persona 1): `corpus_glaciares/papers.csv`, `pdfs/` (398 PDFs) y `pdfs_manifest.csv`
**Salida:** `texto_glaciares/`, el **"corpus procesado"** del hito *Captura con PLN*

| Etapa | Qué hace |
|---|---|
| Lectura | Extrae cada línea con su tipografía y posición (PyMuPDF) |
| Escaneados | Detecta páginas que son solo imagen y, si Tesseract está instalado, las lee con **OCR en español** |
| Limpieza | Quita cabeceras y pies repetidos ("INAIGEM – Dirección de…", "Página 3"), el **índice**, ligaduras y palabras cortadas con guion |
| Estructura | Detecta las secciones de un informe técnico y las normaliza: resumen, introducción, objetivos, área de estudio, metodología, resultados, conclusiones, recomendaciones, referencias, anexos… (también en inglés) |
| Portada | Instituciones y fecha de la portada |
| Tablas y leyendas | Tablas en Markdown (áreas, volúmenes, cotas) y leyendas de figuras, fotos, cuadros y mapas |
| Fragmentos | Trozos de ~1500 caracteres con **procedencia** (documento, sección, páginas) y **pistas del dominio** (laguna, glaciar, peligro, poblado, obra, ubicación, medición) para la Persona 3 |
| Calidad | Reporte automático y plantilla de anotación manual |

## ▶️ Cómo ejecutarlo
1. Abrir en **Jupyter** (o Colab). En la sección 0, las rutas apuntan a la carpeta de Drive de siempre.
2. **Kernel → Restart Kernel and Run All Cells.**
3. Tarda aproximadamente 15–40 minutos la primera vez (398 PDFs); es **reanudable**: si se corta, se vuelve a ejecutar y sigue donde quedó.

### OCR para PDFs escaneados (recomendado)
Varios informes antiguos de la ANA son fotos de páginas, sin texto. Sin OCR, de esos documentos solo se usa el título y el resumen.
- **Windows:** instalar Tesseract desde <https://github.com/UB-Mannheim/tesseract/wiki>. En el instalador, en *Additional language data*, marcar **Spanish**. Luego dejar `RUTA_TESSDATA = r"C:\Program Files\Tesseract-OCR\tessdata"` en la configuración.
- **Colab:** la celda de código lo instala sola.
- Si no se instala, el notebook funciona igual y el reporte indica cuántos documentos quedaron solo con resumen.

ℹ️ El mensaje *"Consider using the pymupdf_layout package…"* es solo una sugerencia de la librería; se puede ignorar.

## 0. Carpetas

In [1]:
# Carpetas en Jupyter (Google Drive para escritorio)
CARPETA_CORPUS_LOCAL = r"G:\Mi unidad\proyecto_gc\corpus_glaciares"   # salida de la Persona 1
CARPETA_TEXTO_LOCAL  = r"G:\Mi unidad\proyecto_gc\texto_glaciares"    # salida de este notebook

import os, sys
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/proyecto_gc"
    CARPETA_CORPUS = os.path.join(BASE, "corpus_glaciares")
    CARPETA_TEXTO = os.path.join(BASE, "texto_glaciares")
else:
    CARPETA_CORPUS, CARPETA_TEXTO = CARPETA_CORPUS_LOCAL, CARPETA_TEXTO_LOCAL
CARPETA_PDFS = os.path.join(CARPETA_CORPUS, "pdfs")
os.makedirs(CARPETA_TEXTO, exist_ok=True)
print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter local")
print("Entrada:", CARPETA_CORPUS)
print("Salida: ", CARPETA_TEXTO)

Mounted at /content/drive
Entorno: Google Colab
Entrada: /content/drive/MyDrive/proyecto_gc/corpus_glaciares
Salida:  /content/drive/MyDrive/proyecto_gc/texto_glaciares


## 1. Configuración ✏️

In [2]:
CONFIG = {
    "CHUNK_OBJETIVO": 1500,      # caracteres por fragmento (~350 tokens)
    "CHUNK_MAXIMO":   2200,      # párrafos más largos se dividen por oraciones
    "SOLAPE":          200,      # caracteres que se repiten entre fragmentos consecutivos
    "EXTRAER_TABLAS": True,      # tablas en Markdown
    "MAX_PAGINAS_TABLAS": 80,    # solo en las primeras N páginas (los inventarios largos tardan mucho)
    "USAR_OCR":       True,      # leer páginas escaneadas con Tesseract (si está instalado)
    "MAX_PAGINAS_OCR": 40,       # máximo de páginas con OCR por documento (~3 s por página)
    "FORZAR":         False,     # True = reprocesa todo (p. ej. tras cambiar el código)
    "N_ANOTACION":       12,     # documentos para la plantilla de anotación manual
}
# Carpeta "tessdata" de Tesseract en Windows (vacío = detectar sola)
RUTA_TESSDATA = r"C:\Program Files\Tesseract-OCR\tessdata"

## 2. Código
Solo se ejecuta; no hace falta modificarla.

In [3]:
%pip install -q -U pymupdf pandas
import os, sys
if "google.colab" in sys.modules:   # en Colab se instala Tesseract con español
    os.system("apt-get -qq install -y tesseract-ocr tesseract-ocr-spa > /dev/null")

# -*- coding: utf-8 -*-
"""
Persona 2 — Del PDF al texto estructurado (caso Glaciares)
Proyecto: Enterprise Brain — Gestión del Conocimiento (Knowledge Graph + GraphRAG)

Entrada (Persona 1): papers.csv + pdfs/ + pdfs_manifest.csv  (398 documentos INAIGEM, ANA, SENAMHI)
Para cada PDF:
  1. Lee cada línea con su tipografía y posición (PyMuPDF).
  2. Detecta páginas escaneadas y, si Tesseract está instalado, les aplica OCR (español).
  3. Limpia: cabeceras y pies repetidos, números de página, índice / tabla de contenido,
     ligaduras y palabras cortadas con guion.
  4. Detecta secciones de informes técnicos en español (y de artículos en inglés) y las normaliza:
     resumen, introduccion, objetivos, antecedentes, area_estudio, metodologia, resultados,
     discusion, conclusiones, recomendaciones, referencias, anexos...
  5. Separa la primera página (institución, fecha), leyendas (Figura, Foto, Cuadro, Mapa) y tablas.
  6. Divide el texto en fragmentos con procedencia (documento, sección, páginas) y marca
     "pistas" del dominio (laguna, glaciar, peligro, poblado, obra, ubicación, medición)
     para que la Persona 3 priorice la extracción de entidades y relaciones.
Es reanudable: no reprocesa los PDFs que no cambiaron.
"""
import os, re, json, gzip, hashlib, unicodedata
from collections import Counter
from datetime import datetime, timezone

import pandas as pd

try:
    import pymupdf
except ImportError:  # versiones antiguas
    import fitz as pymupdf

VERSION = "2.0-glaciares"


def ahora():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFD", t or "") if unicodedata.category(c) != "Mn")


def normalizar_texto(t):
    t = unicodedata.normalize("NFKC", t or "")  # ligaduras ﬁ -> fi, etc.
    t = t.replace("­", "").replace("​", "").replace("﻿", "")
    return re.sub(r"[ \t]+", " ", t).strip()


def nombre_archivo(paper_id):
    """Mismo nombre que usa la Persona 1 para el PDF (válido en Windows)."""
    return re.sub(r"[^\w.-]", "_", str(paper_id))


# ======================================================================
# 1. OCR (opcional, requiere Tesseract instalado)
# ======================================================================
def preparar_ocr(usar_ocr=True, ruta_tesseract=""):
    """Devuelve (disponible, idioma, mensaje)."""
    if not usar_ocr:
        return False, "", "OCR desactivado en la configuración"
    if ruta_tesseract:
        os.environ["TESSDATA_PREFIX"] = ruta_tesseract
    try:
        tessdata = os.environ.get("TESSDATA_PREFIX") or pymupdf.get_tessdata()
    except Exception:
        tessdata = ""
    if not tessdata or not os.path.isdir(tessdata):
        return False, "", ("Tesseract no está instalado: los PDFs escaneados usarán el resumen de los metadatos. "
                           "Ver la sección 0 para instalarlo.")
    os.environ["TESSDATA_PREFIX"] = tessdata
    idiomas = {f[:-12] for f in os.listdir(tessdata) if f.endswith(".traineddata")}
    if "spa" in idiomas:
        return True, "spa+eng" if "eng" in idiomas else "spa", f"OCR disponible (español) en {tessdata}"
    if "eng" in idiomas:
        return True, "eng", f"OCR disponible, pero sin el idioma español (spa.traineddata) en {tessdata}: se usará inglés"
    return False, "", f"Tesseract sin idiomas en {tessdata}"


# ======================================================================
# 2. Lectura de líneas con tipografía
# ======================================================================
def es_negrita(span):
    fuente = (span.get("font") or "").lower()
    return bool(span.get("flags", 0) & 16) or any(k in fuente for k in ("bold", "black", "heavy", "semibold", "negrita", ",b", "-b"))


def unir_spans(spans):
    """Une los tramos de una línea; si hay separación visual sin espacio, agrega uno."""
    out, x_prev = "", None
    for sp in spans:
        t = sp.get("text", "")
        if not t:
            continue
        x0 = sp["bbox"][0]
        if out and x_prev is not None and not out.endswith(" ") and not t.startswith(" ") \
                and x0 - x_prev > 0.15 * sp.get("size", 10):
            out += " "
        out += t
        x_prev = sp["bbox"][2]
    return out


def lineas_de_pagina(page, pno, textpage=None):
    lineas = []
    W, H = page.rect.width, page.rect.height
    d = page.get_text("dict", flags=pymupdf.TEXT_MEDIABOX_CLIP, textpage=textpage)
    for bi, b in enumerate(d.get("blocks", [])):
        if b.get("type") != 0:
            continue
        for li, ln in enumerate(b.get("lines", [])):
            dx, dy = ln.get("dir", (1, 0))
            if abs(dy) > 0.5:  # texto vertical en el margen
                continue
            spans = [s for s in ln.get("spans", []) if s.get("text", "").strip()]
            if not spans:
                continue
            texto = normalizar_texto(unir_spans(ln["spans"]))
            if not texto:
                continue
            principal = max(spans, key=lambda s: len(s["text"].strip()))
            x0, y0, x1, y1 = ln["bbox"]
            lineas.append({
                "page": pno, "block": bi, "line": li, "text": texto,
                "size": round(principal.get("size", 0), 1),
                "bold": sum(len(s["text"]) for s in spans if es_negrita(s)) >= 0.6 * sum(len(s["text"]) for s in spans),
                "x0": x0, "y0": y0, "x1": x1, "y1": y1, "W": W, "H": H,
            })
    return lineas


def leer_documento(doc, ocr=(False, ""), max_paginas_ocr=40, min_car_pagina=80):
    """Lee todas las páginas. Las páginas sin texto pero con imágenes se marcan como escaneadas
    y, si hay OCR, se leen con Tesseract. Devuelve (lineas, info)."""
    ocr_ok, idioma = ocr
    lineas, escaneadas, con_ocr, errores_ocr = [], [], [], 0
    for pno, page in enumerate(doc):
        lp = lineas_de_pagina(page, pno)
        n_car = sum(len(l["text"]) for l in lp)
        if n_car < min_car_pagina and page.get_images(full=False):
            escaneadas.append(pno + 1)
            if ocr_ok and len(con_ocr) < max_paginas_ocr:
                try:
                    tp = page.get_textpage_ocr(language=idioma, dpi=200, full=True)
                    lp_ocr = lineas_de_pagina(page, pno, textpage=tp)
                    if sum(len(l["text"]) for l in lp_ocr) > n_car:
                        for l in lp_ocr:
                            l["ocr"] = True
                        lp = lp_ocr
                        con_ocr.append(pno + 1)
                except Exception:
                    errores_ocr += 1
        lineas += lp
    return lineas, {"paginas_escaneadas": escaneadas, "paginas_ocr": con_ocr, "errores_ocr": errores_ocr}


def quitar_cabeceras_pies(lineas, n_paginas, margen=0.09, min_frac=0.35):
    """Elimina líneas en los márgenes que se repiten en muchas páginas
    (p. ej. 'INAIGEM - Dirección de Investigación en Glaciares') y números de página."""
    def clave(l):
        return re.sub(r"\d+", "#", sin_tildes(l["text"].lower()))

    def en_margen(l):
        return l["y1"] < l["H"] * margen or l["y0"] > l["H"] * (1 - margen)

    conteo = {}
    for l in lineas:
        if en_margen(l):
            conteo.setdefault(clave(l), set()).add(l["page"])
    umbral = max(2, int(round(n_paginas * min_frac)))
    repetidas = {k for k, pags in conteo.items() if len(pags) >= umbral}
    salida, quitadas = [], 0
    for l in lineas:
        if en_margen(l) and (clave(l) in repetidas or
                             re.fullmatch(r"(p[aá]g(ina)?\.?\s*)?[\divxlc\-–— /de]{1,12}", l["text"].lower())):
            quitadas += 1
            continue
        salida.append(l)
    return salida, quitadas


# ======================================================================
# 3. Índice (tabla de contenido)
# ======================================================================
PAT_LINEA_INDICE = re.compile(r"[A-Za-zÁÉÍÓÚÑáéíóúñ].*?(\.{3,}|…{2,}|(\s\.){3,}|_{3,})\s*\d{1,3}\s*$")
PAT_FIN_NUMERO = re.compile(r"[A-Za-zÁÉÍÓÚÑáéíóúñ)]\s+\d{1,3}\s*$")


def paginas_indice(lineas, max_pagina=10):
    """Páginas (entre las primeras) donde la mayoría de líneas terminan en número de página."""
    por_pag = {}
    for l in lineas:
        if l["page"] < max_pagina:
            por_pag.setdefault(l["page"], []).append(l["text"])
    out = set()
    for p, textos in por_pag.items():
        n = sum(bool(PAT_LINEA_INDICE.search(t) or PAT_FIN_NUMERO.search(t)) for t in textos)
        cabecera = any(re.match(r"^(indice|contenido|tabla de contenido|indice general)\b", sin_tildes(t.lower()))
                       for t in textos[:6])
        if n >= 5 and (n >= 0.4 * len(textos) or cabecera):
            out.add(p)
    return out


# ======================================================================
# 4. Encabezados y tipos de sección (español + inglés)
# ======================================================================
PAT_NUMERADO = re.compile(
    r"^(?P<num>\d{1,2}(?:\.\d{1,2}){0,3}|[A-H](?:\.\d{1,2}){0,3}|[IVX]{1,5})[.)\-–]?\s+"
    r"(?P<titulo>[A-ZÁÉÍÓÚÑÜ¿(][^\n]{1,110})$")
PAT_SOLO_NUMERO = re.compile(r"^(\d{1,2}(?:\.\d{1,2}){0,3}|[IVX]{1,5})[.)]?$")
PAT_CAPITULO = re.compile(r"^cap[ií]tulo\s+(?P<num>[ivxlc\d]+)\b\.?\s*[:\-–.]?\s*(?P<titulo>.*)$", re.I)
PAT_LEYENDA = re.compile(
    r"^(figura|fig\.|foto(graf[ií]a)?|imagen|tabla|cuadro|gr[aá]fico|mapa|l[aá]mina|plano|esquema|"
    r"figure|table|photo)\s*(n\s*[°º.]\s*|nro\.?\s*|no\.?\s*)?\d+[a-z]?(\.\d+)?\s*[:.\-–|]", re.I)

NOMBRES_CONOCIDOS = {
    # español
    "resumen", "resumen ejecutivo", "presentacion", "prologo", "introduccion", "objetivo", "objetivos",
    "objetivo general", "objetivos especificos", "antecedentes", "marco teorico", "marco conceptual",
    "justificacion", "alcance", "alcances", "ubicacion", "ubicacion y acceso", "accesibilidad",
    "area de estudio", "zona de estudio", "ambito de estudio", "descripcion del area de estudio",
    "metodologia", "materiales y metodos", "metodos", "trabajos de campo", "trabajo de campo",
    "resultados", "resultados y discusion", "discusion", "analisis", "evaluacion", "diagnostico",
    "conclusiones", "conclusion", "conclusiones y recomendaciones", "recomendaciones",
    "referencias", "referencias bibliograficas", "bibliografia", "literatura citada",
    "anexos", "anexo", "apendice", "panel fotografico", "registro fotografico", "agradecimientos",
    "indice", "contenido", "tabla de contenido", "glosario", "abreviaturas", "siglas y acronimos",
    # inglés (artículos científicos del corpus)
    "abstract", "summary", "introduction", "background", "related work", "study area", "data and methods",
    "methods", "methodology", "materials and methods", "data", "results", "discussion", "conclusion",
    "conclusions", "acknowledgments", "acknowledgements", "references", "appendix", "supplementary material",
}

TIPOS_REGEX = [
    ("indice", r"^(indice|contenido|tabla de contenido|indice general|lista de (figuras|tablas|cuadros|fotos|fotografias|mapas|anexos)|"
               r"indice de (figuras|tablas|cuadros|fotografias|mapas|anexos)|glosario|abreviaturas|siglas)"),
    ("agradecimientos", r"agradecimiento|acknowledg"),
    ("referencias", r"^(referencias|bibliografia|literatura citada|fuentes consultadas|references|bibliography)"),
    ("anexos", r"^(anexo|apendice|appendi|panel fotografico|registro fotografico|supplementa)"),
    ("resumen", r"^(resumen|resumen ejecutivo|abstract|summary|sumario)$"),
    ("presentacion", r"^(presentacion|prologo|prefacio|foreword|preface)"),
    ("introduccion", r"^(introduc|generalidades)"),
    ("objetivos", r"objetivo|^finalidad|^alcances?$|^justificacion"),
    ("antecedentes", r"antecedente|marco teorico|marco conceptual|estado del arte|background|related work|literature"),
    ("area_estudio", r"ubicacion|area de estudio|zona de estudio|ambito|accesibilidad|localizacion|study area|"
                     r"descripcion (general )?(de la|del) (zona|area|laguna|glaciar|cuenca|subcuenca|nevado)|"
                     r"caracteristicas (generales|fisicas|de la laguna|del glaciar|de la cuenca)|geografia|clima de la zona"),
    ("conclusiones", r"^conclusi|conclusion"),
    ("recomendaciones", r"recomendaci"),
    ("discusion", r"^discusi|discussion"),
    ("metodologia", r"metodolog|metodo|materiales y|procedimiento|trabajos? de campo|methods?\b|methodology|"
                    r"^data\b|datos y|levantamiento|instrumentacion|equipos? utilizados"),
    ("resultados", r"resultado|analisis|evaluacion|diagnostico|caracterizacion|inspeccion|monitoreo|"
                   r"inventario|batimetri|peligro|riesgo|results|evaluation|analysis"),
]
TIPOS_SIN_FRAGMENTOS = ("indice", "referencias", "agradecimientos", "front_matter")


def tipo_de_titulo(titulo):
    t = re.sub(r"[^a-z ]", " ", sin_tildes(titulo.lower()))
    t = re.sub(r"\s+", " ", t).strip()
    for tipo, pat in TIPOS_REGEX:
        if re.search(pat, t):
            return tipo
    return "otro"


def limpiar_titulo(t):
    return re.sub(r"\s+", " ", t).strip(" .:-–")


def detectar_encabezado(l, cuerpo):
    """Devuelve (numero, titulo, nivel) si la línea es un encabezado; si no, None."""
    texto = l["text"].strip()
    if len(texto) > 120 or len(texto.split()) > 16 or PAT_LEYENDA.match(texto):
        return None
    letras = re.sub(r"[^A-Za-zÁÉÍÓÚÑÜáéíóúñü]", "", texto)
    mayusculas = len(letras) >= 4 and letras.isupper()
    destacado = l["bold"] or l["size"] >= cuerpo + 0.8 or mayusculas
    m = PAT_CAPITULO.match(texto)
    if m and (destacado or len(texto.split()) <= 6):
        return m.group("num").upper(), limpiar_titulo(m.group("titulo")) or texto, 1
    sin_num = limpiar_titulo(re.sub(r"^(\d{1,2}(?:\.\d{1,2}){0,3}|[A-H]|[IVX]{1,5})[.)\-–]?\s+", "", texto))
    if sin_tildes(sin_num.lower()) in NOMBRES_CONOCIDOS and (destacado or l["size"] >= cuerpo - 0.2):
        num = texto[: len(texto) - len(sin_num)].strip(" .)-–") if texto.lower() != sin_num.lower() else ""
        return num, sin_num, (num.count(".") + 1) if num else 1
    m = PAT_NUMERADO.match(texto)
    if m and destacado:
        num, titulo = m.group("num"), limpiar_titulo(m.group("titulo"))
        if re.match(r"^[A-H]$", num) and not (l["bold"] or mayusculas):
            return None
        if titulo.endswith(".") and len(titulo.split()) > 7:
            return None  # oración numerada (lista de conclusiones o recomendaciones)
        if sum(c.isdigit() for c in titulo) > 6:
            return None
        nivel = num.count(".") + 1 if num[0].isdigit() else 1
        return num, titulo, nivel
    return None


# ======================================================================
# 5. Procesamiento de un PDF
# ======================================================================
PAT_INSTITUCION = re.compile(
    r"\b(INAIGEM|ANA|SENAMHI|INDECI|CENEPRED|INGEMMET|IGP|INRENA|ELECTROPER[UÚ]|HIDRANDINA|EGENOR|"
    r"autoridad nacional del agua|instituto\s+\w+|servicio nacional|ministerio|direcci[oó]n|subdirecci[oó]n|"
    r"unidad de glaciolog\w*|gobierno regional|municipalidad|universi\w+|facultad|"
    r"university|institute|department|laborator\w+|centro de)\b", re.I)
PAT_EMAIL = re.compile(r"[\w.+-]+@[\w-]+(\.[\w-]+)+")
PAT_FECHA = re.compile(r"\b(enero|febrero|marzo|abril|mayo|junio|julio|agosto|se(p)?tiembre|octubre|noviembre|diciembre)"
                       r"\s*(de|del)?\s*(19|20)\d{2}\b", re.I)

PISTAS = {
    "laguna": r"\blagunas?\b|\blake\b|\blakes\b",
    "glaciar": r"\bglaciar\w*|\bnevados?\b|\bglacier\w*|\bcobertura glaciar|\bmasa de hielo",
    "peligro": r"riesgo|peligr\w*|aluvi[oó]n\w*|desborde|glof|avalanch\w*|huaico|huayco|deslizamiento|vulnerab\w*|hazard|outburst",
    "poblado": r"centros? poblados?|ciudad de|distrito de|localidad|caser[ií]o|comunidad campesina|poblaci[oó]n|habitantes|viviendas",
    "obra": r"obras? de (seguridad|control)|\bdique\w*|t[uú]nel\w*|canal de (desag[uü]e|descarga)|sif[oó]n\w*|aliviadero|"
            r"represamiento|presa de|desembalse|sistema de alerta temprana",
    "ubicacion": r"cordillera|subcuenca|microcuenca|cuenca del?|departamento|provincia de|regi[oó]n (de )?[A-ZÁÉÍÓÚ]|quebrada",
    "medicion": r"\d+(?:[.,]\d+)?\s?(?:m\.?\s?s\.?\s?n\.?\s?m\.?|msnm|km2|km²|m2|m²|m3|m³|hm3|hm³|mmc|ha\b|%|m/a|m/año)",
}
PISTAS_RE = {k: re.compile(v, re.I) for k, v in PISTAS.items()}
STOP_ES = set("de la que el en los del se las por un para con una su al lo como más pero sus le ya o fue este ha sí porque esta son entre "
              "cuando muy sin sobre también me hasta hay donde quien desde todo nos durante todos uno les ni contra otros ese eso".split())
STOP_EN = set("the of and to in is that for it as with was on be by this are from at or an which have has were not but their these".split())


def idioma(texto):
    pal = re.findall(r"[a-záéíóúñ]+", texto[:20000].lower())
    es, en = sum(p in STOP_ES for p in pal), sum(p in STOP_EN for p in pal)
    if es + en < 20:
        return "desconocido"
    return "es" if es >= en else "en"


def unir_lineas(textos):
    """Une líneas de un párrafo reparando palabras cortadas con guion."""
    out = ""
    for t in textos:
        if not out:
            out = t
        elif out.endswith("-") and t[:1].islower() and not re.search(r"-\w+-$", out):
            out = out[:-1] + t
        else:
            out += " " + t
    return out.strip()


def procesar_pdf(ruta, paper_id, titulo_meta="", abstract_meta="", extraer_tablas=True,
                 max_paginas_tablas=80, ocr=(False, ""), max_paginas_ocr=40):
    res = {"paper_id": paper_id, "version": VERSION, "title": titulo_meta, "flags": [], "procesado": ahora()}
    try:
        doc = pymupdf.open(ruta)
    except Exception as e:
        res["flags"].append("pdf_no_abre")
        res["error"] = str(e)[:300]
        return res
    res["n_pages"] = doc.page_count

    lineas, info = leer_documento(doc, ocr, max_paginas_ocr)
    res.update(info)
    res["ocr_aplicado"] = bool(info["paginas_ocr"])
    res["ocr_disponible"] = bool(ocr[0])
    n_esc = len(info["paginas_escaneadas"])
    if n_esc:
        res["flags"].append(f"escaneado ({n_esc}/{doc.page_count} pág.)" if n_esc >= 0.5 * doc.page_count
                            else f"paginas_escaneadas ({n_esc})")
    if n_esc and not info["paginas_ocr"]:
        res["flags"].append("sin_ocr")

    # Tablas (antes del texto para no mezclar sus celdas con los párrafos). No en páginas OCR.
    tablas, zonas_tabla = [], {}
    if extraer_tablas:
        for pno in range(min(doc.page_count, max_paginas_tablas)):
            if pno + 1 in info["paginas_escaneadas"]:
                continue
            try:
                for t in doc[pno].find_tables().tables:
                    md = t.to_markdown(clean=True) if hasattr(t, "to_markdown") else ""
                    if md and md.count("\n") >= 2:
                        tablas.append({"page": pno + 1, "bbox": [round(v, 1) for v in t.bbox], "markdown": md})
                        zonas_tabla.setdefault(pno, []).append(t.bbox)
            except Exception:
                pass

    texto_total = " ".join(l["text"] for l in lineas)
    res["pdf_text_chars"] = len(texto_total)
    if texto_total.count("(cid:") > 20 or texto_total.count("�") > 50:
        res["flags"].append("texto_con_errores_de_codificacion")
    res["idioma"] = idioma(texto_total)

    lineas, n_quitadas = quitar_cabeceras_pies(lineas, doc.page_count)
    res["lineas_cabecera_pie_quitadas"] = n_quitadas
    pags_indice = paginas_indice(lineas)
    res["paginas_indice"] = sorted(p + 1 for p in pags_indice)

    def en_tabla(l):
        for (x0, y0, x1, y1) in zonas_tabla.get(l["page"], []):
            if l["x0"] >= x0 - 2 and l["x1"] <= x1 + 2 and l["y0"] >= y0 - 2 and l["y1"] <= y1 + 2:
                return True
        return False

    tamanos = Counter()
    for l in lineas:
        tamanos[l["size"]] += len(l["text"])
    cuerpo = tamanos.most_common(1)[0][0] if tamanos else 10.0
    res["font_size_cuerpo"] = cuerpo

    # Título: línea(s) con la mayor tipografía en la primera página (portada)
    p1 = [l for l in lineas if l["page"] == 0]
    titulo_ids = set()
    if p1:
        mayor = max(l["size"] for l in p1)
        if not titulo_meta:
            res["title"] = " ".join(l["text"] for l in p1 if l["size"] == mayor)[:300]
        if mayor >= cuerpo + 2:
            titulo_ids = {(l["page"], l["block"], l["line"]) for l in p1 if l["size"] == mayor}

    secciones = [{"heading": "PORTADA", "num": "", "level": 0, "type": "front_matter", "parrafos": []}]
    leyendas, lineas_front, n_lineas_indice = [], [], 0
    buffer, buf_key, buf_pag = [], None, None
    pendiente_num = None

    def cerrar_parrafo():
        nonlocal buffer, buf_key, buf_pag
        if buffer:
            texto = unir_lineas(buffer)
            if PAT_LEYENDA.match(texto) and len(texto) < 400:
                leyendas.append({"page": buf_pag + 1, "text": texto, "section": secciones[-1]["heading"]})
            elif len(texto) > 1:
                previos = secciones[-1]["parrafos"]
                if previos and previos[-1]["text"].endswith("-") and texto[:1].islower():
                    previos[-1]["text"] = previos[-1]["text"][:-1] + texto   # palabra cortada entre bloques
                else:
                    previos.append({"page": buf_pag + 1, "text": texto})
        buffer, buf_key, buf_pag = [], None, None

    for l in lineas:
        if (l["page"], l["block"], l["line"]) in titulo_ids or en_tabla(l):
            continue
        if l["page"] in pags_indice or PAT_LINEA_INDICE.search(l["text"]):
            n_lineas_indice += 1
            continue
        if pendiente_num is not None:
            l = dict(l, text=f"{pendiente_num} {l['text']}")
            pendiente_num = None
        if PAT_SOLO_NUMERO.match(l["text"]) and (l["bold"] or l["size"] >= cuerpo + 0.8):
            cerrar_parrafo()
            pendiente_num = l["text"].strip(" .)")
            continue
        enc = detectar_encabezado(l, cuerpo)
        if enc:
            cerrar_parrafo()
            num, titulo, nivel = enc
            secciones.append({"heading": titulo, "num": num, "level": nivel, "type": tipo_de_titulo(titulo),
                              "page_start": l["page"] + 1, "parrafos": []})
            continue
        if len(secciones) == 1:
            lineas_front.append(l["text"])
        clave = (l["page"], l["block"])
        if clave != buf_key:
            cerrar_parrafo()
            buf_key, buf_pag = clave, l["page"]
        buffer.append(l["text"])
    cerrar_parrafo()
    res["lineas_indice_quitadas"] = n_lineas_indice

    # Tipos: herencia en subsecciones y deducción por posición
    padre = None
    for s in secciones[1:]:
        if s["level"] <= 1:
            padre = s
        elif s["type"] == "otro" and padre is not None:
            s["type"] = padre["type"]
    i_intro = next((i for i, s in enumerate(secciones) if s["type"] in ("introduccion", "objetivos", "antecedentes")), None)
    i_fin = next((i for i, s in enumerate(secciones) if s["type"] in ("conclusiones", "recomendaciones", "referencias")), None)
    i_ref = next((i for i, s in enumerate(secciones) if s["type"] == "referencias"), None)
    for i, s in enumerate(secciones):
        if s["type"] != "otro":
            continue
        if i_ref is not None and i > i_ref:
            s["type"] = "anexos"
        elif i_intro is not None and i > i_intro and (i_fin is None or i < i_fin):
            s["type"] = "desarrollo"   # capítulo propio del informe (p. ej. "Descripción de peligros")
    if i_ref is not None:
        for s in secciones[i_ref + 1:]:
            if s["type"] not in ("agradecimientos", "referencias", "anexos", "conclusiones", "recomendaciones"):
                s["tipo_original"] = s["type"]
                s["type"] = "anexos"

    # Resumen: sección del PDF o, si no hay, el de los metadatos (lo normal en informes técnicos)
    resumen_sec = next((s for s in secciones if s["type"] == "resumen"), None)
    resumen = " ".join(p["text"] for p in resumen_sec["parrafos"]) if resumen_sec else (abstract_meta or "")
    res["resumen_fuente"] = "pdf" if resumen_sec else ("metadatos" if abstract_meta else "no_encontrado")

    front_texto = "\n".join(p["text"] for p in secciones[0]["parrafos"])
    p1_texto = "\n".join(l["text"] for l in lineas if l["page"] == 0)
    candidatas = []
    for ln in lineas_front[:60] + p1_texto.split("\n")[:60]:
        if PAT_INSTITUCION.search(ln) and not PAT_EMAIL.search(ln) and 3 < len(ln) < 200:
            candidatas.append(ln.strip(" ,"))
    fechas = [m.group(0) for m in PAT_FECHA.finditer(front_texto + "\n" + p1_texto)]
    res.update({
        "resumen": resumen, "front_matter": front_texto[:5000],
        "instituciones_candidatas": list(dict.fromkeys(candidatas))[:20],
        "fechas_portada": list(dict.fromkeys(fechas))[:5],
        "emails": list(dict.fromkeys(m.group(0) for m in PAT_EMAIL.finditer(front_texto)))[:20],
        "first_page_text": p1_texto[:6000],
        "captions": leyendas, "tables": tablas,
    })

    salida = []
    for s in secciones:
        texto = "\n\n".join(p["text"] for p in s["parrafos"])
        if s["type"] == "front_matter" and not texto:
            continue
        paginas = [p["page"] for p in s["parrafos"]] or [s.get("page_start", 1)]
        salida.append({"idx": len(salida), "heading": s["heading"], "num": s["num"], "level": s["level"],
                       "type": s["type"], "page_start": min(paginas), "page_end": max(paginas),
                       "n_chars": len(texto), "parrafos": s["parrafos"], "text": texto})
    res["sections"] = salida

    tipos = {s["type"] for s in salida}
    principales = [s for s in salida if s["level"] == 1 and s["type"] != "front_matter"]
    sin_estructura = len(principales) < 3
    excluir = ("indice", "referencias", "agradecimientos") if sin_estructura else TIPOS_SIN_FRAGMENTOS
    n_util = sum(s["n_chars"] for s in salida if s["type"] not in excluir)
    if n_util < 1500:
        res["flags"].append("sin_texto_util")
    elif sin_estructura:
        res["flags"].append("estructura_no_detectada")
    res["tipos_detectados"] = sorted(tipos)
    res["n_sections"] = len(salida)
    res["n_chars_texto"] = sum(s["n_chars"] for s in salida)
    res["n_chars_util"] = n_util
    doc.close()
    return res


# ======================================================================
# 6. Fragmentación con procedencia y pistas del dominio
# ======================================================================
def dividir_oraciones(texto):
    return [o for o in re.split(r"(?<=[.!?;])\s+(?=[A-ZÁÉÍÓÚÑ¿¡(\[\d])", texto) if o.strip()]


def pistas(texto):
    return [k for k, p in PISTAS_RE.items() if p.search(texto)]


def fragmentar(res, meta=None, objetivo=1500, maximo=2200, solape=200):
    """Fragmentos por sección respetando párrafos. Sin estructura: todo el cuerpo.
    Sin texto útil (escaneado sin OCR): un único fragmento con título + resumen de los metadatos."""
    meta = meta or {}
    pid = res["paper_id"]
    base = {"paper_id": pid, "fuente": meta.get("fuente", ""), "title": meta.get("title", res.get("title", "")),
            "year": meta.get("year", "")}
    chunks = []
    secciones = res.get("sections", [])
    flags = res.get("flags", [])
    if "sin_texto_util" in flags or not secciones:
        texto = (base["title"] + ". " + (res.get("resumen") or meta.get("abstract", "") or "")).strip()
        if len(texto) > 20:
            p = pistas(texto)
            chunks.append({**base, "chunk_id": f"{pid}::meta::c000", "section_idx": -1, "section_type": "resumen",
                           "heading": "Resumen (metadatos)", "page_start": None, "page_end": None,
                           "text": texto, "n_chars": len(texto), "pistas": p, "prioridad": len(p),
                           "fuente_texto": "metadatos"})
        return chunks
    if "estructura_no_detectada" in flags:
        todos = [p for s in secciones for p in s["parrafos"] if s["type"] not in ("indice", "referencias", "agradecimientos")]
        secciones = [{"idx": 0, "heading": "CUERPO", "type": "cuerpo", "parrafos": todos}]
    paginas_ocr = set(res.get("paginas_ocr", []))
    for s in secciones:
        if s["type"] in TIPOS_SIN_FRAGMENTOS:
            continue
        piezas = []
        for p in s["parrafos"]:
            if len(p["text"]) <= maximo:
                piezas.append(p)
            else:
                actual = ""
                for o in dividir_oraciones(p["text"]):
                    while len(o) > maximo:   # oración enorme (texto OCR sin puntos)
                        piezas.append({"page": p["page"], "text": o[:objetivo]})
                        o = o[objetivo:]
                    if actual and len(actual) + len(o) + 1 > objetivo:
                        piezas.append({"page": p["page"], "text": actual})
                        actual = o
                    else:
                        actual = (actual + " " + o).strip()
                if actual:
                    piezas.append({"page": p["page"], "text": actual})
        texto, paginas, k = "", [], 0

        def emitir():
            nonlocal k
            t = texto.strip()
            if len(t) >= 40:
                p = pistas(t)
                chunks.append({
                    **base, "chunk_id": f"{pid}::s{s['idx']:02d}::c{k:03d}", "section_idx": s["idx"],
                    "section_type": s["type"], "heading": s["heading"],
                    "page_start": min(paginas), "page_end": max(paginas), "text": t, "n_chars": len(t),
                    "pistas": p, "prioridad": len([x for x in p if x != "medicion"]) + ("medicion" in p) * 0.5,
                    "fuente_texto": "ocr" if set(paginas) & paginas_ocr else "pdf",
                })
                k += 1

        for p in piezas:
            if texto and len(texto) + len(p["text"]) + 2 > objetivo:
                emitir()
                arrastre = ""
                for o in reversed(dividir_oraciones(texto)):
                    if len(arrastre) + len(o) > solape:
                        break
                    arrastre = (o + " " + arrastre).strip()
                texto, paginas = arrastre, paginas[-1:]
            texto = (texto + "\n\n" + p["text"]).strip()
            paginas.append(p["page"])
        emitir()
    return chunks


# ======================================================================
# 7. Proceso por lotes (reanudable)
# ======================================================================
def ruta_pdf_de(pid, carpeta_pdfs, manifiesto_p1=None):
    if manifiesto_p1 is not None and pid in manifiesto_p1.index:
        archivo = manifiesto_p1.loc[pid, "archivo"]
        if isinstance(archivo, str) and archivo:
            return os.path.join(carpeta_pdfs, archivo)
    return os.path.join(carpeta_pdfs, nombre_archivo(pid) + ".pdf")


def procesar_corpus(papers, carpeta_pdfs, carpeta_salida, manifiesto_p1=None, extraer_tablas=True,
                    max_paginas_tablas=80, ocr=(False, ""), max_paginas_ocr=40, forzar=False, log=print):
    carpeta_json = os.path.join(carpeta_salida, "json")
    os.makedirs(carpeta_json, exist_ok=True)
    ruta_man = os.path.join(carpeta_salida, "texto_manifest.csv")
    previo = (pd.read_csv(ruta_man, dtype={"paper_id": str}).drop_duplicates("paper_id", keep="last").set_index("paper_id")
              if os.path.exists(ruta_man) else None)
    filas, resultados = [], {}
    total = len(papers)
    for i, f in enumerate(papers.itertuples(), start=1):
        pid = str(f.paper_id)
        ruta_pdf = ruta_pdf_de(pid, carpeta_pdfs, manifiesto_p1)
        ruta_json = os.path.join(carpeta_json, nombre_archivo(pid) + ".json")
        if not os.path.exists(ruta_pdf):
            filas.append({"paper_id": pid, "estado": "sin_pdf", "flags": "sin_pdf", "procesado": ahora()})
            log(f"   ❌ {pid}: no está el PDF ({os.path.basename(ruta_pdf)})")
            continue
        sha = sha256_archivo(ruta_pdf)
        reutilizar = (not forzar and previo is not None and pid in previo.index and os.path.exists(ruta_json)
                      and previo.loc[pid, "sha256_pdf"] == sha and str(previo.loc[pid, "version"]) == VERSION)
        res = None
        if reutilizar:
            with open(ruta_json, encoding="utf-8") as fh:
                res = json.load(fh)
            # si antes no había OCR y ahora sí, se reprocesan los escaneados
            if ocr[0] and res.get("paginas_escaneadas") and not res.get("ocr_disponible"):
                reutilizar, res = False, None
        if res is None:
            res = procesar_pdf(ruta_pdf, pid, getattr(f, "title", "") or "", getattr(f, "abstract", "") or "",
                               extraer_tablas, max_paginas_tablas, ocr, max_paginas_ocr)
            res["pdf_sha256"] = sha
            with open(ruta_json, "w", encoding="utf-8") as fh:
                json.dump(res, fh, ensure_ascii=False)
        resultados[pid] = res
        flags = res.get("flags", [])
        estado = ("error" if "pdf_no_abre" in flags else
                  "solo_metadatos" if "sin_texto_util" in flags else
                  "ok_ocr" if res.get("ocr_aplicado") else "ok")
        filas.append({"paper_id": pid, "estado": estado, "sha256_pdf": sha, "version": VERSION,
                      "n_pages": res.get("n_pages"), "idioma": res.get("idioma", ""),
                      "paginas_escaneadas": len(res.get("paginas_escaneadas", [])),
                      "paginas_ocr": len(res.get("paginas_ocr", [])),
                      "n_sections": res.get("n_sections", 0), "n_chars_util": res.get("n_chars_util", 0),
                      "resumen_fuente": res.get("resumen_fuente", ""),
                      "n_tablas": len(res.get("tables", [])), "n_leyendas": len(res.get("captions", [])),
                      "tipos_detectados": ";".join(res.get("tipos_detectados", [])),
                      "flags": ";".join(flags), "procesado": res.get("procesado", ahora()),
                      "reutilizado": reutilizar})
        if i % 10 == 0 or i == total:
            log(f"   {i} / {total} procesados")
            pd.DataFrame(filas).to_csv(ruta_man, index=False, encoding="utf-8")
    manifiesto = pd.DataFrame(filas)
    manifiesto.to_csv(ruta_man, index=False, encoding="utf-8")
    return resultados, manifiesto


# ======================================================================
# 8. Salidas, reporte y documentación
# ======================================================================
def _jsonl(ruta, filas, comprimir=False):
    abrir = gzip.open if comprimir else open
    with abrir(ruta, "wt", encoding="utf-8") as fh:
        for x in filas:
            fh.write(json.dumps(x, ensure_ascii=False, default=str) + "\n")


def construir_salidas(resultados, papers, carpeta_salida, objetivo=1500, maximo=2200, solape=200):
    meta = papers.set_index("paper_id")
    carpeta_txt = os.path.join(carpeta_salida, "textos")
    os.makedirs(carpeta_txt, exist_ok=True)
    secciones, chunks, primeras, leyendas, tablas, docs = [], [], [], [], [], []
    for pid, r in resultados.items():
        m = meta.loc[pid].to_dict() if pid in meta.index else {}
        m = {k: ("" if (isinstance(v, float) and pd.isna(v)) else v) for k, v in m.items()}
        for s in r.get("sections", []):
            secciones.append({"paper_id": pid, "section_idx": s["idx"], "heading": s["heading"], "num": s["num"],
                              "level": s["level"], "section_type": s["type"], "page_start": s["page_start"],
                              "page_end": s["page_end"], "n_chars": s["n_chars"], "text": s["text"]})
        ch = fragmentar(r, m, objetivo, maximo, solape)
        chunks += ch
        # Texto limpio completo (sin índice ni referencias), para leer y para el RAG
        cuerpo = "\n\n".join(f"## {s['heading']}\n\n{s['text']}" for s in r.get("sections", [])
                             if s["type"] not in ("indice", "referencias") and s["text"])
        if not cuerpo:
            cuerpo = "\n\n".join(c["text"] for c in ch)
        with open(os.path.join(carpeta_txt, nombre_archivo(pid) + ".txt"), "w", encoding="utf-8") as fh:
            fh.write(f"# {m.get('title', r.get('title', ''))}\n\nFuente: {m.get('fuente', '')} · Año: {m.get('year', '')} · "
                     f"paper_id: {pid}\n\n{cuerpo}\n")
        primeras.append({"paper_id": pid, "title": m.get("title", ""), "fuente": m.get("fuente", ""),
                         "instituciones_candidatas": r.get("instituciones_candidatas", []),
                         "fechas_portada": r.get("fechas_portada", []), "emails": r.get("emails", []),
                         "first_page_text": r.get("first_page_text", "")})
        for c in r.get("captions", []):
            leyendas.append({"paper_id": pid, **c, "tipo": sin_tildes(c["text"].split()[0].lower().rstrip(".:"))})
        for k, t in enumerate(r.get("tables", [])):
            tablas.append({"table_id": f"{pid}::t{k:02d}", "paper_id": pid, **t})
        docs.append({"paper_id": pid, "title": m.get("title", ""), "fuente": m.get("fuente", ""), "year": m.get("year", ""),
                     "eje_principal": m.get("eje_principal", ""), "idioma": r.get("idioma", ""),
                     "n_pages": r.get("n_pages"), "n_chars_util": r.get("n_chars_util", 0),
                     "fuente_texto": ("metadatos" if ch and ch[0]["fuente_texto"] == "metadatos"
                                      else "pdf+ocr" if r.get("ocr_aplicado") else "pdf"),
                     "n_secciones": r.get("n_sections", 0), "n_fragmentos": len(ch),
                     "n_tablas": len(r.get("tables", [])), "resumen_fuente": r.get("resumen_fuente", ""),
                     "secciones": ";".join(r.get("tipos_detectados", [])), "observaciones": ";".join(r.get("flags", []))})
    df_sec = pd.DataFrame(secciones)
    df_chunks = pd.DataFrame(chunks)
    df_docs = pd.DataFrame(docs)
    df_sec.to_csv(os.path.join(carpeta_salida, "sections.csv"), index=False, encoding="utf-8")
    df_docs.to_csv(os.path.join(carpeta_salida, "documentos_procesados.csv"), index=False, encoding="utf-8")
    pd.DataFrame(leyendas).to_csv(os.path.join(carpeta_salida, "captions.csv"), index=False, encoding="utf-8")
    _jsonl(os.path.join(carpeta_salida, "chunks.jsonl"), chunks)
    _jsonl(os.path.join(carpeta_salida, "chunks.jsonl.gz"), chunks, comprimir=True)   # para GitHub (pesa menos)
    _jsonl(os.path.join(carpeta_salida, "first_pages.jsonl"), primeras)
    _jsonl(os.path.join(carpeta_salida, "tables.jsonl"), tablas)
    return df_docs, df_sec, df_chunks, pd.DataFrame(primeras), pd.DataFrame(leyendas), pd.DataFrame(tablas)


TIPOS_CLAVE = ["resumen", "introduccion", "objetivos", "area_estudio", "metodologia", "resultados", "conclusiones",
               "recomendaciones", "referencias"]


def reporte_extraccion(manifiesto, df_docs, df_sec, df_chunks, leyendas, tablas, n_docs, ocr_msg=""):
    ok, warn, err = "✅", "⚠️", "❌"
    chk = []
    estados = manifiesto["estado"].value_counts().to_dict()
    con_texto = estados.get("ok", 0) + estados.get("ok_ocr", 0)
    chk.append((ok if con_texto >= 0.9 * n_docs else warn,
                f"Documentos con texto del PDF: {con_texto}/{n_docs} (de ellos con OCR: {estados.get('ok_ocr', 0)})"))
    solo_meta = estados.get("solo_metadatos", 0)
    chk.append((ok if solo_meta == 0 else warn,
                f"Documentos solo con título + resumen (escaneados sin OCR): {solo_meta}"))
    sin_pdf = estados.get("sin_pdf", 0) + estados.get("error", 0)
    chk.append((ok if sin_pdf == 0 else err, f"Documentos sin PDF o con PDF dañado: {sin_pdf}"))
    sin_frag = n_docs - (df_chunks.paper_id.nunique() if len(df_chunks) else 0)
    chk.append((ok if sin_frag == 0 else err, f"Documentos sin ningún fragmento: {sin_frag}"))
    flags = manifiesto["flags"].fillna("")
    estructura = flags.str.contains("estructura_no_detectada").sum()
    chk.append((ok if estructura <= n_docs * 0.25 else warn,
                f"Documentos sin secciones detectadas (se fragmenta el cuerpo completo): {estructura}"))
    cobertura = {}
    for t in TIPOS_CLAVE:
        cobertura[t] = int(df_sec[df_sec.section_type == t].paper_id.nunique()) if len(df_sec) else 0
    if len(df_chunks):
        prior = (df_chunks.prioridad >= 2).sum()
        chk.append((ok if prior >= len(df_chunks) * 0.3 else warn,
                    f"Fragmentos con ≥2 pistas del dominio (prioritarios para la Persona 3): {prior}/{len(df_chunks)}"))
    idiomas = df_docs.idioma.value_counts().to_dict() if len(df_docs) else {}
    lineas = ["# Reporte de extracción de texto (Persona 2 · Glaciares)", "", f"Generado: {ahora()} · versión {VERSION}",
              "", f"OCR: {ocr_msg}", "", "| Estado | Verificación |", "|---|---|"] + [f"| {e} | {m} |" for e, m in chk]
    lineas += ["", "## Corpus procesado", "",
               f"- Páginas: {int(manifiesto.n_pages.fillna(0).sum())} · caracteres útiles: {int(manifiesto.n_chars_util.fillna(0).sum()):,}",
               f"- Idioma: {idiomas}",
               "- Por fuente: " + (", ".join(f"{k}: {v}" for k, v in df_docs.fuente.value_counts().items()) if len(df_docs) else ""),
               "- Documentos con cada sección: " + ", ".join(f"{k}: {v}" for k, v in cobertura.items())]
    if len(df_chunks):
        conteo_p = Counter(p for ps in df_chunks.pistas for p in ps)
        lineas += ["", "## Fragmentos (chunks)", "",
                   f"- Total: {len(df_chunks)} · promedio por documento: {len(df_chunks) / max(n_docs, 1):.1f}",
                   f"- Tamaño medio: {df_chunks.n_chars.mean():.0f} caracteres",
                   "- Por tipo de sección: " + ", ".join(f"{k}: {v}" for k, v in df_chunks.section_type.value_counts().items()),
                   "- Por origen del texto: " + ", ".join(f"{k}: {v}" for k, v in df_chunks.fuente_texto.value_counts().items()),
                   "- Fragmentos con cada pista: " + ", ".join(f"{k}: {v}" for k, v in conteo_p.most_common())]
    lineas += ["", "## Otros", "", f"- Leyendas (figuras, fotos, cuadros, mapas): {len(leyendas)}",
               f"- Tablas extraídas: {len(tablas)}", f"- Secciones totales: {len(df_sec)}"]
    problemas = manifiesto[flags != ""][["paper_id", "flags"]]
    if len(problemas):
        lineas += ["", "## Documentos con observaciones", "", "| paper_id | observaciones |", "|---|---|"]
        lineas += [f"| {r.paper_id} | {r.flags} |" for r in problemas.itertuples()]
    return chk, "\n".join(lineas), cobertura


def plantilla_anotacion(papers, n=12, semilla=7):
    """Muestra estratificada por fuente para anotar a mano (gold standard).
    Sirve para medir la extracción de la Persona 3 y adelanta la evaluación de la semana 12."""
    p = papers.copy()
    partes = [g.sample(min(len(g), max(1, n // p["fuente"].nunique())), random_state=semilla) for _, g in p.groupby("fuente")]
    m = pd.concat(partes)
    if len(m) < n:
        m = pd.concat([m, p[~p.paper_id.isin(m.paper_id)].sample(min(n - len(m), len(p) - len(m)), random_state=semilla)])
    cols = ["lagunas", "glaciares_nevados", "cordillera_cuenca_departamento", "nivel_de_riesgo", "eventos_y_fechas",
            "centros_poblados_expuestos", "obras_de_seguridad", "metodos_de_monitoreo", "instituciones",
            "mediciones_clave", "anotador", "notas"]
    salida = m[["paper_id", "title", "fuente"]].head(n).copy()
    for c in cols:
        salida[c] = ""
    return salida


README_TEXTO = """# Texto procesado del corpus Glaciares (Persona 2)

Entrada: `corpus_glaciares/papers.csv`, `pdfs/` y `pdfs_manifest.csv` (Persona 1).
Herramienta: PyMuPDF (tipografía y posición) + Tesseract para páginas escaneadas (opcional).

## Archivos

| Archivo | Contenido | Lo usa |
|---|---|---|
| `documentos_procesados.csv` | Una fila por documento: fuente, idioma, páginas, origen del texto (pdf, pdf+ocr, metadatos), secciones, fragmentos | Entrega del hito, control |
| `textos/<id>.txt` | Texto limpio completo de cada documento, con sus secciones | Lectura, revisión |
| `chunks.jsonl` (`.gz`) | Fragmentos de ~1500 caracteres con procedencia (documento, sección, páginas) y pistas del dominio | **Persona 3 (entidades y relaciones)** y RAG (semana 9) |
| `sections.csv` | Una fila por sección: tipo normalizado, título, páginas, texto | Persona 3, análisis |
| `tables.jsonl` | Tablas en Markdown (inventarios, áreas, volúmenes, cotas) | Persona 3 (valores numéricos) |
| `captions.csv` | Leyendas de figuras, fotos, cuadros y mapas | Persona 3 |
| `first_pages.jsonl` | Portada: instituciones candidatas, fechas | Persona 3 (publicadoPor, fechas) |
| `texto_manifest.csv` | Estado por PDF, hash, páginas escaneadas y con OCR, observaciones | Control de calidad |
| `reporte_extraccion.md` | Verificaciones automáticas | Entrega del hito |
| `anotacion_manual_plantilla.csv` | 12 documentos para anotar a mano (gold standard) | Evaluación de la extracción |

## Tipos de sección

`resumen, presentacion, introduccion, objetivos, antecedentes, area_estudio, metodologia, resultados,
desarrollo, discusion, conclusiones, recomendaciones, referencias, anexos, agradecimientos, indice,
front_matter, cuerpo, otro`

- `desarrollo`: capítulos propios del informe entre la introducción y las conclusiones
  (p. ej. "Descripción de los peligros", "Laguna Palcacocha").
- `cuerpo`: documento sin estructura detectable; se fragmenta completo.
- El índice, las referencias y los agradecimientos no se fragmentan.

## Campos de `chunks.jsonl`

`chunk_id` (`<documento>::s<sección>::c<n>`), `paper_id`, `fuente`, `title`, `year`, `section_idx`,
`section_type`, `heading`, `page_start`, `page_end`, `text`, `n_chars`, `pistas`, `prioridad`, `fuente_texto`.

- `pistas`: qué menciona el fragmento: `laguna`, `glaciar`, `peligro`, `poblado`, `obra`, `ubicacion`, `medicion`.
- `prioridad`: número de pistas (medición cuenta 0.5). Los fragmentos con prioridad ≥ 2 concentran los hechos
  que responden las preguntas de competencia; la Persona 3 puede empezar por ellos.
- `fuente_texto`: `pdf`, `ocr` (página escaneada leída con Tesseract) o `metadatos` (solo título y resumen).

## Uso de los textos

Los documentos son publicaciones de acceso abierto de instituciones públicas peruanas (INAIGEM, ANA, SENAMHI).
Se cita siempre la fuente con `paper_id` (handle del repositorio) y las páginas del fragmento.
"""


print("Persona 2 cargada · versión", VERSION, "· PyMuPDF", pymupdf.VersionBind)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 65.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
Persona 2 cargada · versión 2.0-glaciares · PyMuPDF 1.28.2


## 3. Verificar la entrada y el OCR

In [4]:
papers = pd.read_csv(os.path.join(CARPETA_CORPUS, "papers.csv"), dtype={"paper_id": str})
ruta_man_p1 = os.path.join(CARPETA_CORPUS, "pdfs_manifest.csv")
man_p1 = (pd.read_csv(ruta_man_p1, dtype={"paper_id": str}).drop_duplicates("paper_id", keep="last").set_index("paper_id")
          if os.path.exists(ruta_man_p1) else None)

faltan = [p for p in papers.paper_id if not os.path.exists(ruta_pdf_de(p, CARPETA_PDFS, man_p1))]
print(f"Documentos en el corpus: {len(papers)} | con PDF: {len(papers) - len(faltan)}")
print("Por fuente:", papers.fuente.value_counts().to_dict())
if faltan:
    print(f"⚠️ {len(faltan)} sin PDF (la Persona 1 debe ejecutar su sección 9):", faltan[:10])
else:
    print("✅ Todos los documentos tienen PDF")

OCR_OK, OCR_IDIOMA, OCR_MSG = preparar_ocr(CONFIG["USAR_OCR"], RUTA_TESSDATA if os.path.isdir(RUTA_TESSDATA) else "")
print(("✅ " if OCR_OK and "spa" in OCR_IDIOMA else "⚠️ ") + OCR_MSG)

Documentos en el corpus: 398 | con PDF: 398
Por fuente: {'INAIGEM': 175, 'ANA': 171, 'SENAMHI': 52}
✅ Todos los documentos tienen PDF
✅ OCR disponible (español) en /usr/share/tesseract-ocr/5/tessdata/


## 4. Procesar los PDFs
Muestra el avance cada 10 documentos. Si se interrumpe, al volver a ejecutar solo procesa los que faltan.

In [5]:
import time
t0 = time.time()
resultados, manifiesto = procesar_corpus(
    papers, CARPETA_PDFS, CARPETA_TEXTO, man_p1,
    extraer_tablas=CONFIG["EXTRAER_TABLAS"], max_paginas_tablas=CONFIG["MAX_PAGINAS_TABLAS"],
    ocr=(OCR_OK, OCR_IDIOMA), max_paginas_ocr=CONFIG["MAX_PAGINAS_OCR"], forzar=CONFIG["FORZAR"])
print(f"\nTiempo: {(time.time() - t0) / 60:.1f} min | reutilizados: {int(manifiesto.reutilizado.fillna(False).sum())}")
print("Estado:", manifiesto.estado.value_counts().to_dict())
print("   ok = texto del PDF · ok_ocr = con páginas leídas por OCR · solo_metadatos = escaneado sin OCR")
print("Idioma:", manifiesto.idioma.value_counts().to_dict())
print(f"Páginas: {int(manifiesto.n_pages.fillna(0).sum())} | escaneadas: {int(manifiesto.paginas_escaneadas.fillna(0).sum())} "
      f"| leídas con OCR: {int(manifiesto.paginas_ocr.fillna(0).sum())}")
obs = manifiesto[manifiesto["flags"].fillna("") != ""]
print(f"\nDocumentos con observaciones: {len(obs)}")
print(obs["flags"].str.split(";").explode().str.replace(r" \(.*", "", regex=True).value_counts().to_string())

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
   10 / 398 procesados
   20 / 398 procesados
   30 / 398 procesados
   40 / 398 procesados
   50 / 398 procesados
   60 / 398 procesados
   70 / 398 procesados
   80 / 398 procesados
   90 / 398 procesados
   100 / 398 procesados
   110 / 398 procesados
   120 / 398 procesados
   130 / 398 procesados
   140 / 398 procesados
   150 / 398 procesados
   160 / 398 procesados
   170 / 398 procesados
   180 / 398 procesados
   190 / 398 procesados
   200 / 398 procesados
   210 / 398 procesados
MuPDF error: syntax error: unknown keyword: '-1.#INF'

MuPDF error: syntax error: unknown keyword: '1.#INF'

MuPDF error: syntax error: unknown keyword: '-1.#INF'

MuPDF error: syntax error: unknown keyword: '1.#INF'

   220 / 398 procesados
   230 / 398 procesados
   240 / 398 procesados
   250 / 398 procesados
   260 / 398 procesados
   270 / 398 procesados
   280 / 398 procesados
   290 / 398 procesados
   300 /

## 5. Inspeccionar un documento 🔍
Sirve para comprobar a ojo que las secciones tienen sentido y **para la demo en vivo**.
Cambien `DOC` por cualquier `paper_id` (por ejemplo, uno con observaciones).

In [6]:
DOC = "20.500.12748/394"   # Informe técnico de inspección A06, laguna Islapata (INAIGEM)

r = resultados.get(DOC)
if r is None:
    DOC = next(iter(resultados)); r = resultados[DOC]
    print("Ese ID no está; se muestra", DOC)
fila = papers.set_index("paper_id").loc[DOC]
print(f"{fila.title[:110]}\n{fila.fuente} · {fila.year} · páginas: {r.get('n_pages')} · idioma: {r.get('idioma')} · "
      f"observaciones: {r['flags'] or 'ninguna'}")
print("Instituciones en portada:", r.get("instituciones_candidatas")[:4], "| fechas:", r.get("fechas_portada"))
print(f"Páginas de índice quitadas: {r.get('paginas_indice')} | escaneadas: {len(r.get('paginas_escaneadas', []))} "
      f"| con OCR: {len(r.get('paginas_ocr', []))}\n")
for s in r.get("sections", []):
    print(f"[{s['type']:15s}] {s['num']:5s} {s['heading'][:50]:50s} p.{s['page_start']}-{s['page_end']} {s['n_chars']:6d} car.")
for t in r.get("tables", [])[:1]:
    print(f"\nTabla (p. {t['page']}):\n{t['markdown'][:700]}")

Informe Técnico de Inspección A06 Laguna Islapata 2021
INAIGEM · 2021 · páginas: 30 · idioma: es · observaciones: ['paginas_escaneadas (1)']
Instituciones en portada: ['PS INAIGEM', 'INSTITUTO NACIONAL DE', 'Instituto Nacional de Investigación en Glaciares', 'y Ecosistemas de Montaña – INAIGEM'] | fechas: ['marzo del 2021']
Páginas de índice quitadas: [4] | escaneadas: 1 | con OCR: 1

[front_matter   ]       PORTADA                                            p.1-3    714 car.
[introduccion   ]       Introducción                                       p.5-5   1497 car.
[desarrollo     ] 1     Aspectos Generales                                 p.5-5      0 car.
[area_estudio   ] 1.1   Ubicación                                          p.5-5    407 car.
[objetivos      ] 1.2   Objetivo                                           p.5-5     97 car.
[resultados     ] 2     Caracterización                                    p.5-16   8625 car.
[resultados     ] 3     Criterios para la evaluación 

## 6. Generar salidas y reporte de calidad

In [7]:
df_docs, df_sec, df_chunks, primeras, leyendas, tablas = construir_salidas(
    resultados, papers, CARPETA_TEXTO, CONFIG["CHUNK_OBJETIVO"], CONFIG["CHUNK_MAXIMO"], CONFIG["SOLAPE"])
checks, reporte, cobertura = reporte_extraccion(manifiesto, df_docs, df_sec, df_chunks, leyendas, tablas,
                                                len(papers), OCR_MSG)
with open(os.path.join(CARPETA_TEXTO, "reporte_extraccion.md"), "w", encoding="utf-8") as f:
    f.write(reporte)
with open(os.path.join(CARPETA_TEXTO, "README_texto.md"), "w", encoding="utf-8") as f:
    f.write(README_TEXTO)
ruta_anot = os.path.join(CARPETA_TEXTO, "anotacion_manual_plantilla.csv")
if not os.path.exists(ruta_anot):   # no se sobrescribe si ya empezaron a anotar
    plantilla_anotacion(papers, CONFIG["N_ANOTACION"]).to_csv(ruta_anot, index=False, encoding="utf-8")
    print("Plantilla de anotación creada.")

for e, m in checks:
    print(e, m)
print("\nDocumentos con cada sección:", cobertura)
print(f"Secciones: {len(df_sec)} | fragmentos: {len(df_chunks)} | tablas: {len(tablas)} | leyendas: {len(leyendas)}")
print("Archivos en", CARPETA_TEXTO, ":", sorted(os.listdir(CARPETA_TEXTO)))

Plantilla de anotación creada.
✅ Documentos con texto del PDF: 366/398 (de ellos con OCR: 180)
⚠️ Documentos solo con título + resumen (escaneados sin OCR): 32
✅ Documentos sin PDF o con PDF dañado: 0
✅ Documentos sin ningún fragmento: 0
✅ Documentos sin secciones detectadas (se fragmenta el cuerpo completo): 48
✅ Fragmentos con ≥2 pistas del dominio (prioritarios para la Persona 3): 9633/27394

Documentos con cada sección: {'resumen': 121, 'introduccion': 206, 'objetivos': 143, 'area_estudio': 179, 'metodologia': 153, 'resultados': 205, 'conclusiones': 222, 'recomendaciones': 114, 'referencias': 177}
Secciones: 18910 | fragmentos: 27394 | tablas: 4209 | leyendas: 8471
Archivos en /content/drive/MyDrive/proyecto_gc/texto_glaciares : ['README_texto.md', 'anotacion_manual_plantilla.csv', 'captions.csv', 'chunks.jsonl', 'chunks.jsonl.gz', 'documentos_procesados.csv', 'first_pages.jsonl', 'json', 'reporte_extraccion.md', 'sections.csv', 'tables.jsonl', 'texto_manifest.csv', 'textos']


## 7. Resumen y demostración para la presentación
- El **fragmento de ejemplo** es lo que recibirá la Persona 3 (y luego el RAG): texto + procedencia (documento, sección, páginas). Es la base para que el agente **cite sus fuentes**.
- `buscar()` encuentra en segundos dónde se menciona algo en los 398 documentos (útil como demo en vivo).

In [8]:
print(f"Documentos procesados: {len(df_docs)} | páginas: {int(df_docs.n_pages.fillna(0).sum())} | "
      f"caracteres útiles: {int(df_docs.n_chars_util.sum()):,}")
print("Origen del texto:", df_docs.fuente_texto.value_counts().to_dict())
print(f"Fragmentos: {len(df_chunks)} (media {df_chunks.n_chars.mean():.0f} car.) | "
      f"prioritarios (≥2 pistas): {(df_chunks.prioridad >= 2).sum()}")
print("Fragmentos por pista:", dict(Counter(p for ps in df_chunks.pistas for p in ps).most_common()))

ej = df_chunks.sort_values(["prioridad", "n_chars"], ascending=False)
ej = ej[ej.section_type.isin(["resultados", "conclusiones", "desarrollo", "area_estudio"])].head(1)
ej = (ej if len(ej) else df_chunks.head(1)).iloc[0]
print("\nEjemplo de fragmento:")
print(json.dumps({k: ej[k] for k in ["chunk_id", "paper_id", "fuente", "year", "section_type", "heading",
                                     "page_start", "page_end", "pistas", "prioridad"]},
                 ensure_ascii=False, default=str, indent=1))
print(ej["text"][:700])


def buscar(termino, n=8):
    # Fragmentos que mencionan un término (sin distinguir tildes ni mayúsculas)
    clave = sin_tildes(termino.lower())
    m = df_chunks[df_chunks.text.map(lambda t: clave in sin_tildes(t.lower()))]
    print(f"'{termino}': {len(m)} fragmentos en {m.paper_id.nunique()} documentos")
    with pd.option_context("display.max_colwidth", 90):
        display(m.sort_values("prioridad", ascending=False)[["paper_id", "fuente", "year", "section_type",
                                                              "page_start", "title"]].head(n))

buscar("Palcacocha")

Documentos procesados: 398 | páginas: 19452 | caracteres útiles: 26,172,588
Origen del texto: {'pdf': 186, 'pdf+ocr': 180, 'metadatos': 32}
Fragmentos: 27394 (media 994 car.) | prioritarios (≥2 pistas): 9633
Fragmentos por pista: {'ubicacion': 9050, 'glaciar': 8897, 'medicion': 6682, 'laguna': 5744, 'peligro': 4149, 'poblado': 3986, 'obra': 1397}

Ejemplo de fragmento:
{
 "chunk_id": "20.500.12543/5679::s06::c006",
 "paper_id": "20.500.12543/5679",
 "fuente": "ANA",
 "year": "2023",
 "section_type": "area_estudio",
 "heading": "Dimensiones de la laguna Parón",
 "page_start": 8.0,
 "page_end": 9.0,
 "pistas": [
  "laguna",
  "glaciar",
  "peligro",
  "poblado",
  "obra",
  "ubicacion",
  "medicion"
 ],
 "prioridad": 6.5
}
Es así que el ingeniero, ha determinado que la laguna Parón debe su formación a la unión de un cono aluvial y una morrena, ver Fig. 04

Es sabido que tanto los conos aluviales y las morrenas, son materiales heterogéneos conformado por bloques, gravas, arenas, limos, di

,paper_id,fuente,year,section_type,page_start,title
16848,20.500.12543/2919,ANA,2003,antecedentes,4.0,Informe de evaluación del estado de los glaciares en la cabecera de la laguna Palcacocha
18834,20.500.12543/2623,ANA,2018,desarrollo,82.0,El inventario nacional de glaciares: las cordilleras glaciares del Perú
13748,20.500.12543/4718,ANA,2020,cuerpo,41.0,Lagunas reservas de agua dulce en Áncash: Resultados de estudios de batimetría en 38 l...
4066,20.500.12748/608,INAIGEM,2024,resultados,46.0,Evaluación nacional de lagunas glaciares con riesgo de desborde 2024.
13723,20.500.12543/4718,ANA,2020,cuerpo,15.0,Lagunas reservas de agua dulce en Áncash: Resultados de estudios de batimetría en 38 l...
4069,20.500.12748/608,INAIGEM,2024,resultados,49.0,Evaluación nacional de lagunas glaciares con riesgo de desborde 2024.
9028,20.500.12748/288,INAIGEM,2016,antecedentes,2.0,Informe técnico de la situación actual de la Laguna Glaciar Palcacocha - Enero
11563,20.500.12748/57,INAIGEM,2018,desarrollo,82.0,Inventario Nacional de Glaciares Las Cordilleras Glaciares del Perú 2018


## Entrega
**Al repositorio (GitHub), carpeta `data/persona2/`:** este notebook, `documentos_procesados.csv`, `chunks.jsonl.gz`,
`sections.csv`, `tables.jsonl`, `captions.csv`, `first_pages.jsonl`, `texto_manifest.csv`, `reporte_extraccion.md`,
`README_texto.md` y `anotacion_manual_plantilla.csv`.
- `chunks.jsonl.gz` es el **corpus procesado**; se sube comprimido para no pasar el límite de 100 MB de GitHub.
- No se suben `json/` (caché) ni los PDFs. `textos/` puede ir en Drive o comprimida en un `.zip` si pesa menos de 100 MB.

**Para la Persona 3:** `chunks.jsonl` (empezar por `prioridad >= 2`), `tables.jsonl` y `first_pages.jsonl`.
Cada tripleta que extraiga debe guardar el `chunk_id` (y así el documento y las páginas) de donde salió.

**Anotación manual:** 12 documentos en `anotacion_manual_plantilla.csv` para llenar a mano. Servirán para medir la extracción de la Persona 3.